# m37 lab · An ECG classifier from its parts

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You build the pieces of an ECG classifier, train two small models, and then move them somewhere new to see what
breaks. Parts A–D use the course's synthetic ECG, so the truth is known. Part E uses real public recordings from
PhysioNet. Part F repeats the network in PyTorch, if you have it.

| Part | What you do | Data |
| --- | --- | --- |
| A | Check the softmax gradient numerically; plot a kernel's frequency response | synthetic |
| B | Check the receptive-field formula by perturbing inputs; count beats per output | synthetic |
| C | A "hidden" patient-level signal: compare record-wise and patient-wise splits | synthetic |
| D | Train an RR-interval model and a NumPy CNN for AF; test both on inverted signals and on irregular rhythms that are not AF | synthetic |
| E | Train on the PhysioNet/CinC Challenge 2017 data; test unchanged on PTB-XL lead I; correct for prevalence | PhysioNet |
| F | Optional: the same CNN in PyTorch | synthetic |

The synthetic ECG is an approximate model: a sum of Gaussian waves per beat, after the morphology part of
McSharry et al. (2003). Its wave sizes, times and widths, heart rates and noise levels are illustrative values,
not physiological reference values. AF beats have no P wave and irregular intervals; fibrillatory waves are not
modelled.

Each part ends with **checks**: lines that pass with the code as given. If you change something and a check
fails, that is information, not an error to hide. The lab runs in about two minutes, including downloads.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from epml import aiecg as ai
from epml import ml
from epml.data import DataUnavailable

FS = 100.0  # Hz, the sampling rate used for the synthetic parts

## Part A · Probabilities, gradients and filters

The softmax turns scores into probabilities (eq-m37-softmax), and the cross-entropy gradient is $p - y$
(eq-m37-ce-grad). Check the gradient against finite differences. Then treat a convolution kernel as a filter:
its gain at frequency $f$ is $|H(f)|$ (eq-m37-kernel-response).

In [ ]:
rng = np.random.default_rng(37)
z = rng.normal(size=(1, 4))
y = np.eye(4)[[1]]
h = 1e-6
numeric = np.array([(ai.cross_entropy(z + h * np.eye(4)[[c]], y) - ai.cross_entropy(z - h * np.eye(4)[[c]], y)) / (2 * h)
                    for c in range(4)])
print("p - y     :", np.round(ai.ce_grad(z, y)[0], 6))
print("numerical :", np.round(numeric, 6))

kernels = {"difference [1, 0, -1]/2": np.array([1.0, 0.0, -1.0]) / 2,
           "moving average, 9 taps": np.ones(9) / 9}
f = np.linspace(0, FS / 2, 200)
fig, ax = plt.subplots()
for name, w in kernels.items():
    ax.plot(f, np.abs(ai.kernel_response(w, FS, f)), label=name)
ax.set(xlabel="frequency (Hz)", ylabel="gain |H(f)|", title=f"Two kernels as filters at {FS:.0f} Hz")
ax.legend()
plt.show()

# measured gain on a 10-Hz sinusoid
n = np.arange(3000)
sine = np.cos(2 * np.pi * 10 * n / FS)
w = kernels["moving average, 9 taps"]
measured = np.sqrt(2 * np.mean(ai.conv1d(sine, w)[100:-100] ** 2))
print(f"moving average at 10 Hz: measured gain {measured:.4f}, |H| {abs(ai.kernel_response(w, FS, 10)[0]):.4f}")

In [ ]:
# Checks
assert np.allclose(numeric, ai.ce_grad(z, y)[0], atol=1e-7)
assert abs(measured - abs(ai.kernel_response(w, FS, 10)[0])) < 1e-3
assert abs(ai.kernel_response(kernels["difference [1, 0, -1]/2"], FS, 0)[0]) < 1e-12   # blocks DC

## Part B · What one output can see

The receptive field of a stack of convolutions is $r = 1 + \sum_l (k_l - 1)\prod_{j<l} s_j$ samples
(eq-m37-receptive-field). Check it by perturbing one input sample at a time and watching the first output. Then
convert it to seconds and to RR intervals at 70 beats per minute.

In [ ]:
designs = {"4 x width 9, strides 1,2,2,2": ([9] * 4, [1, 2, 2, 2]),
           "8 x width 9, stride 1": ([9] * 8, [1] * 8),
           "6 x width 15, strides 1,2,2,2,2,2": ([15] * 6, [1, 2, 2, 2, 2, 2])}
rows = []
for name, (k, s) in designs.items():
    r = ai.receptive_field(k, s)
    seen, _ = ai.empirical_receptive_field(k, s)
    for fs in [100, 300, 500]:
        rows.append((name, fs, r, seen, r / fs, r / fs / (60 / 70)))
print(f"{'design':36s} {'fs':>4s} {'r':>5s} {'seen':>5s} {'seconds':>8s} {'RR at 70 bpm':>13s}")
for name, fs, r, seen, sec, beats in rows:
    print(f"{name:36s} {fs:4d} {r:5d} {seen:5d} {sec:8.2f} {beats:13.2f}")

In [ ]:
# Checks
assert all(r == seen for _, _, r, seen, _, _ in rows)
assert ai.receptive_field([9] * 4, [1, 2, 2, 2]) == 65
# the same network sees less time at a higher sampling rate
assert rows[0][4] > rows[1][4] > rows[2][4]

## Part C · A hidden signal, and the price of a record-wise split

A network can find signals no reader uses, such as signs of AF in a sinus-rhythm ECG (Attia et al. 2019). Here is
a toy version. All synthetic patients are in sinus rhythm, but in the "positive" group the T wave is 10% smaller
(an illustrative, made-up effect). Each patient has five records and their own heart rate and amplitude.

Features per record: mean RR, R amplitude, and the T/R ratio. A 1-nearest-neighbour model can recognise a patient
from the first two. Compare a record-wise split, where one patient's records fall on both sides, with a
patient-wise split (m35).

In [ ]:
def hidden_signal_aucs(seed):
    rng = np.random.default_rng(seed)
    X, y, pid, peaks = ai.make_cohort(60, 5, rhythms=("sinus", "sinus"), t_factors=(1.0, 0.9), rng=rng)
    F = np.array([np.r_[ai.rr_features(p, FS)[0], ai.wave_amplitudes(x, p, FS)] for x, p in zip(X, peaks)])
    Z = np.c_[F[:, 0], F[:, 1], F[:, 2] / F[:, 1]]
    record_split = rng.random(len(y)) < 0.67
    patient_split = np.isin(pid, rng.permutation(60)[:40])
    out = {}
    for name, train in [("record-wise", record_split), ("patient-wise", patient_split)]:
        Zs = (Z - Z[train].mean(0)) / Z[train].std(0)
        model = KNeighborsClassifier(1).fit(Zs[train], y[train])
        out[name] = ml.auc_rank(model.predict_proba(Zs[~train])[:, 1], y[~train])
    return out

results = [hidden_signal_aucs(seed) for seed in range(6)]
rec = np.array([r["record-wise"] for r in results])
pat = np.array([r["patient-wise"] for r in results])
print("1-NN test AUC, six cohorts")
print("  record-wise :", np.round(rec, 2), " mean", round(rec.mean(), 3))
print("  patient-wise:", np.round(pat, 2), " mean", round(pat.mean(), 3))

In [ ]:
# Checks
assert rec.mean() - pat.mean() > 0.08      # the record-wise split flatters the model
assert pat.mean() > 0.5                     # but the hidden signal is real

## Part D · Two AF models, two ways to fail

Build a cohort of sinus and AF patients. Model 1 uses RR features (eq-m37-rr) found with a simple QRS detector
and logistic regression (eq-m37-logistic). Model 2 is a small CNN written in NumPy: 8 filters of width 31
(0.31 s at 100 Hz), ReLU, global average pooling and a logistic output, trained with the Adam update. Train
both on a patient-wise split.

Then test both on two kinds of data they never saw:

- the same test patients with **inverted** signals, as when a handheld device is held the other way round;
- patients with frequent early beats (**ectopy**): irregular, but not AF, and with P waves.

In [ ]:
rng = np.random.default_rng(370)
X, y, pid, peaks = ai.make_cohort(80, 3, fs=FS, duration_s=10, rng=rng)
train = np.isin(pid, rng.permutation(80)[:54])
test = ~train
Xs = ai.standardize_records(X)
Xe, ye, _, _ = ai.make_cohort(30, 1, fs=FS, rhythms=("ectopy",), rng=rng)

def rr_matrix(signals, fs=FS, sign=1):
    return np.array([ai.rr_features(ai.detect_r_peaks(sign * x, fs), fs)[1:] for x in signals])

F = rr_matrix(X)
rr_model = LogisticRegression(max_iter=1000).fit(F[train], y[train])
rr_report = {"AUC": ml.auc_rank(rr_model.predict_proba(F[test])[:, 1], y[test]),
             "AUC inverted": ml.auc_rank(rr_model.predict_proba(rr_matrix(X[test], sign=-1))[:, 1], y[test]),
             "ectopy called AF": float((rr_model.predict_proba(rr_matrix(Xe))[:, 1] > 0.5).mean())}

def train_cnn(seed, augment=False, epochs=40):
    """Train the NumPy CNN once; with augment=True, half the training signals are flipped at random."""
    r = np.random.default_rng(seed)
    flip = np.where(r.random(train.sum()) < 0.5, -1.0, 1.0)[:, None] if augment else 1.0
    net = ai.TinyCNN(n_filters=8, width=31, rng=np.random.default_rng(seed))
    loss = net.fit(Xs[train] * flip, y[train], epochs=epochs, lr=0.02, rng=r, optimizer="adam")
    return net, loss

# three training runs, because one run's answer to "does polarity matter?" varies with the seed
runs = [train_cnn(seed) for seed in (11, 12, 13)]
cnn_report = {"AUC": np.mean([ml.auc_rank(n.predict_proba(Xs[test]), y[test]) for n, _ in runs]),
              "AUC inverted": np.mean([ml.auc_rank(n.predict_proba(-Xs[test]), y[test]) for n, _ in runs]),
              "ectopy called AF": np.mean([(n.predict_proba(ai.standardize_records(Xe)) > 0.5).mean() for n, _ in runs])}
report = {"RR model": rr_report, "NumPy CNN (mean of 3)": cnn_report}
for model, r in report.items():
    print(f"{model:22s}  " + "  ".join(f"{k} {v:.2f}" for k, v in r.items()))

fig, ax = plt.subplots()
for _, loss in runs:
    ax.plot(loss)
ax.set(xlabel="epoch", ylabel="training cross-entropy", title="NumPy CNN training, three runs")
plt.show()

The interval model is blind to polarity but calls most ectopy AF, because irregular intervals are all it
sees. The CNN leaned on waveform shape: it ignores ectopy here (P waves are present) but loses ground when the
signal is flipped. A standard fix for the second problem is **augmentation**: flip half the training signals at
random.

In [ ]:
aug_runs = [train_cnn(seed, augment=True) for seed in (11, 12, 13)]
auc_aug = np.mean([ml.auc_rank(n.predict_proba(Xs[test]), y[test]) for n, _ in aug_runs])
auc_aug_inv = np.mean([ml.auc_rank(n.predict_proba(-Xs[test]), y[test]) for n, _ in aug_runs])
print(f"augmented CNN (mean of 3): AUC {auc_aug:.2f}, inverted {auc_aug_inv:.2f}")

In [ ]:
# Checks
cnn = report["NumPy CNN (mean of 3)"]
assert report["RR model"]["AUC"] > 0.95 and cnn["AUC"] > 0.9
assert abs(report["RR model"]["AUC inverted"] - report["RR model"]["AUC"]) < 0.05   # polarity does not matter
assert report["RR model"]["ectopy called AF"] > 0.5                                 # irregular is not AF
assert cnn["ectopy called AF"] < 0.2
assert cnn["AUC"] - cnn["AUC inverted"] > 0.05                                      # shape-based: polarity matters
assert auc_aug_inv > 0.9

## Part E · Real recordings: train in one place, test in another

Training data: a balanced subset of normal-rhythm (N) and AF (A) recordings from the PhysioNet/CinC Challenge 2017
training set (Clifford et al. 2017): single-lead, 300 Hz, 0.5–40 Hz band, 9–61 s, often inverted. We resample to
100 Hz and keep the first 10 s.

External test: lead I of PTB-XL 1.0.3 (Wagner et al. 2020), 100 Hz copy, 10 s, from fold 10, the paper's
recommended test fold. Records labelled with sinus rhythm (SCP code SR) or AF (AFIB), but not both.

Licenses: Challenge 2017, Open Data Commons Attribution v1.0; PTB-XL, Creative Commons Attribution 4.0. Every file
is checked against the database's published SHA-256 list. Without internet access this part is skipped.

The Challenge data carry no patient identifiers, so the internal split below is by recording. That is one more
reason to trust the external test more than the internal one.

In [ ]:
from scipy.signal import resample_poly

from epml.data import load_challenge2017, load_ptbxl_lead

N_TRAIN, N_TEST = 150, 60      # records per class: small, so the lab stays quick
try:
    sig, lab, names = load_challenge2017(n_per_class=N_TRAIN, classes=("N", "A"))
    ext, ext_lab, ext_ids, ext_counts = load_ptbxl_lead(n_per_class=N_TEST, codes=("SR", "AFIB"), fold=10)
    REAL = True
except DataUnavailable as err:
    print("Part E skipped:", err)
    REAL = False

In [ ]:
if REAL:
    def to_10s_at_100(x, fs_in):
        x = resample_poly(x, 1, 3) if fs_in == 300 else np.asarray(x, float)
        return x[:1000] if len(x) >= 1000 else None

    pairs = [(to_10s_at_100(x, 300), int(l == "A")) for x, l in zip(sig, lab)]
    pairs = [(x, l) for x, l in pairs if x is not None]
    Xc = np.array([x for x, _ in pairs])
    yc = np.array([l for _, l in pairs])
    Xp = np.asarray(ext)[:, :1000]
    yp = np.array([int(l == "AFIB") for l in ext_lab])
    print(f"Challenge: {len(yc)} records ({yc.mean():.0%} AF). PTB-XL fold 10 sample: {len(yp)} records ({yp.mean():.0%} AF).")

    rng = np.random.default_rng(7)
    internal = rng.random(len(yc)) < 0.3
    def rr_rows(signals):
        out = []
        for x in signals:
            f = ai.rr_features(ai.detect_r_peaks(x, 100.0), 100.0)[1:]
            out.append(np.nan_to_num(f, nan=0.0))
        return np.array(out)
    Fc, Fp = rr_rows(Xc), rr_rows(Xp)
    rr_real = LogisticRegression(max_iter=1000).fit(Fc[~internal], yc[~internal])

    Xc_s, Xp_s = ai.standardize_records(Xc), ai.standardize_records(Xp)
    flip = np.where(rng.random((~internal).sum()) < 0.5, -1.0, 1.0)[:, None]
    cnn_real = ai.TinyCNN(n_filters=8, width=31, rng=np.random.default_rng(3))
    cnn_real.fit(Xc_s[~internal] * flip, yc[~internal], epochs=40, lr=0.02, rng=rng, optimizer="adam")

    real = {}
    for name, p_int, p_ext in [
        ("RR model", rr_real.predict_proba(Fc[internal])[:, 1], rr_real.predict_proba(Fp)[:, 1]),
        ("NumPy CNN", cnn_real.predict_proba(Xc_s[internal]), cnn_real.predict_proba(Xp_s)),
    ]:
        slope, intercept = ml.calibration_slope_intercept(np.clip(p_ext, 1e-6, 1 - 1e-6), yp)
        real[name] = {"internal AUC": ml.auc_rank(p_int, yc[internal]), "external AUC": ml.auc_rank(p_ext, yp),
                      "external calibration slope": slope}
    for name, r in real.items():
        print(f"{name:10s}  " + "  ".join(f"{k} {v:.2f}" for k, v in r.items()))

Both samples above are roughly balanced. In PTB-XL fold 10 as a whole, AF is rarer. Correct the RR model's
probabilities for that prevalence with eq-m37-prior-shift, and compare the positive predictive value at a 0.5
threshold before and after.

In [ ]:
if REAL:
    prev_train = yc[~internal].mean()
    prev_fold = ext_counts["AFIB"] / (ext_counts["AFIB"] + ext_counts["SR"])
    p_ext = rr_real.predict_proba(Fp)[:, 1]
    p_adj = ai.prior_shift(p_ext, prev_train, prev_fold)
    for name, p in [("raw", p_ext), ("prevalence-corrected", p_adj)]:
        flag = p > 0.5
        se = flag[yp == 1].mean()
        sp = (~flag[yp == 0]).mean()
        ppv = ml.predictive_values(se, sp, prev_fold)[0] if se > 0 else float("nan")
        print(f"{name:21s} threshold 0.5: sensitivity {se:.2f}, specificity {sp:.2f}, PPV at fold-10 prevalence "
              f"{prev_fold:.1%}: {ppv:.2f}")

In [ ]:
# Checks
if REAL:
    assert len(yc) > 100 and len(yp) > 60
    assert real["RR model"]["external AUC"] > 0.7               # interval features travel between devices
    assert all(0 <= r["external AUC"] <= 1 for r in real.values())
    assert np.allclose(np.argsort(p_adj), np.argsort(p_ext))    # the correction keeps the ranking
    assert 0 < prev_fold < 0.5

## Part F · Optional: the same CNN in PyTorch

Deep-learning libraries compute the gradients for you (automatic differentiation). This is the Part D network in
PyTorch: one convolution layer with 8 filters of width 31, ReLU, global average pooling and a linear output,
trained with the binary cross-entropy. It runs only if PyTorch is installed (`pip install torch`).

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part F skipped.")

if torch is not None:
    torch.manual_seed(0)
    net = torch.nn.Sequential(
        torch.nn.Conv1d(1, 8, kernel_size=31),   # eq-m37-conv, 8 learned filters
        torch.nn.ReLU(),
        torch.nn.AdaptiveAvgPool1d(1),           # global average pooling
        torch.nn.Flatten(),
        torch.nn.Linear(8, 1),                   # one score; sigmoid inside the loss
    )
    Xt = torch.tensor(Xs[:, None, :], dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    tr = torch.tensor(train)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    lossf = torch.nn.BCEWithLogitsLoss()
    for epoch in range(40):
        opt.zero_grad()
        loss_t = lossf(net(Xt[tr]).squeeze(1), yt[tr])
        loss_t.backward()
        opt.step()
    with torch.no_grad():
        p_torch = torch.sigmoid(net(Xt[~tr]).squeeze(1)).numpy()
    auc_torch = ml.auc_rank(p_torch, y[~train])
    print(f"PyTorch CNN: final training loss {loss_t.item():.3f}, test AUC {auc_torch:.3f}")

In [ ]:
# Checks
if torch is not None:
    assert auc_torch > 0.8

## Where to go next

- Change the CNN's width in Part D and rerun Part B's table: when does one output cover two RR intervals?
- In Part E, train on 300-Hz signals without resampling and test on PTB-XL's 500-Hz copy. What happens to the
  CNN, whose filters were learned in samples, not seconds?
- Read the module's evidence ladder again: which rung did this lab reach? (Only the first.)